In [25]:
import os
import warnings
warnings.filterwarnings("ignore")
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()

True

Basic Messaging

In [12]:
embeddings = OpenAIEmbeddings()
llm = ChatOpenAI()
parser = StrOutputParser()
message = "Hi!"

In [13]:
chain = llm | parser
chain.invoke(message)

'Hello! How can I help you today?'

Prompting

In [18]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Translate the following into {language}"),
    ("user", "{text}")
])

In [20]:
prompt = prompt_template.invoke({"language":"french", "text":"I am learning a new Language today!"})
llm.invoke(prompt)

AIMessage(content="Je suis en train d'apprendre une nouvelle langue aujourd'hui !", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 24, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BLgB57oq1yTjFQYv777aSktDnBUQn', 'finish_reason': 'stop', 'logprobs': None}, id='run-fd769c6c-57c5-4869-b88d-1ca5fbd45239-0', usage_metadata={'input_tokens': 24, 'output_tokens': 15, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
chain.invoke(prompt)

"Je suis en train d'apprendre une nouvelle langue aujourd'hui !"

In [22]:
prompt.to_messages()

[SystemMessage(content='Translate the following into french', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am learning a new Language today!', additional_kwargs={}, response_metadata={})]

Chaining

In [23]:
chain = prompt_template | llm | parser
chain.invoke({"language":"hindi", "text":"I am learning a new Language today!"})

'मैं आज एक नई भाषा सीख रहा हूँ!'

Agents

In [ ]:
memory = MemorySaver()
search = TavilySearchResults(max_results=2)
tools = [search]

In [28]:
agent_executor = create_react_agent(llm, tools, checkpointer=memory)

In [29]:
config = {"configurable":{"thread_id":"abc123"}}

In [30]:
for chunk in agent_executor.stream({"messages":[HumanMessage(content="hi im Shubham! and i live in Raleigh")]}, config):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content="Hello Shubham! It's great to meet you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 92, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BLhCacLcYHa8BhGNqNTCg8yxGKaFT', 'finish_reason': 'stop', 'logprobs': None}, id='run-c22cccc6-c5ec-4520-bef2-df3f3575d24b-0', usage_metadata={'input_tokens': 92, 'output_tokens': 21, 'total_tokens': 113, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}}
----


In [31]:
for chunk in agent_executor.stream({"messages":[HumanMessage(content="What's the wheather where I live?")]}, config):
    print(chunk)
    print("----")

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_7nZianooRcTXYlV0KbUFvKF9', 'function': {'arguments': '{"query":"weather in Raleigh"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 128, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-BLhDICcT6WgsdmaDgUXlHvK4L4Tuv', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-d9c174b3-7c22-432a-a096-76a6f5d8840e-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'weather in Raleigh'}, 'id': 'call_7nZianooRcTXYlV0KbUFvKF9', 'type': 'tool_call'}], usage_metadata={'input_tokens': 128, 'output_tokens': 21, 'total_to